In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL_NAME = "allenai_OLMo-2-1124-7B-Instruct"
MODEL_LABEL = "OLMo 2 7B Instruct"

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "figure_notebooks":
    REPO_ROOT = REPO_ROOT.parent
elif REPO_ROOT.name == "syntactic_diversity":
    REPO_ROOT = REPO_ROOT.parent

RESULT_DIR = Path(
    os.environ.get(
        "OLMO2_HUMANEVAL_RESULT_DIR",
        str(REPO_ROOT / "data" / "raw" / "Tokenizer_passK" / "results_passretok" / "humaneval" / MODEL_NAME),
    )
)
FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})

RESULT_DIR

In [ ]:
def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required figure input does not exist: {path}")
    return path


def new_metric_path(path: Path) -> Path:
    """Return the *_new_metric path for a .json or .jsonl metric file."""
    path = Path(path)
    if path.name.endswith("_new_metric.json") or path.name.endswith("_new_metric.jsonl"):
        return path
    if path.suffix not in {".json", ".jsonl"}:
        raise ValueError(f"Expected a JSON/JSONL metric file, got: {path}")
    return path.with_name(f"{path.stem}_new_metric{path.suffix}")


def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(require_file(path), lines=True)


def standard_error(df: pd.DataFrame, std_col: str, n_col: str) -> pd.Series:
    n = df[n_col].where(df[n_col] > 0)
    return df[std_col] / np.sqrt(n)


def save_figure(fig: plt.Figure, stem: str) -> None:
    svg_path = FIGURE_DIR / f"{stem}.svg"
    png_path = FIGURE_DIR / f"{stem}.png"
    fig.savefig(svg_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print(f"Saved {svg_path}")
    print(f"Saved {png_path}")

In [ ]:
RETOK_RESULTS = RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data.df"
CANON_RESULTS = RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data_0.df"
RETOK_METRICS = new_metric_path(RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data.df_synctactic.jsonl")
CANON_METRICS = new_metric_path(RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data_0.df_synctactic.jsonl")
RETOK_CORRECT_METRICS = new_metric_path(RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data.df_synctactic_correct.jsonl")
CANON_CORRECT_METRICS = new_metric_path(RESULT_DIR / "retokp_maxexamples_164_unbiasedsize_51_df_data_0.df_synctactic_correct.jsonl")

In [ ]:
def load_pass_metric_comparison(retok_metric_path: Path, canon_metric_path: Path) -> pd.DataFrame:
    retok = read_jsonl(retok_metric_path).set_index("task_id")
    canon = read_jsonl(canon_metric_path).set_index("task_id").add_suffix("_passk")
    joined = retok.join(canon, how="inner")

    joined["std_error_retok"] = standard_error(joined, "std_syntactic_metric", "num_valid_pairs")
    joined["std_error_passk"] = standard_error(joined, "std_syntactic_metric_passk", "num_valid_pairs_passk")
    return joined


def correctness_by_task(path: Path) -> pd.Series:
    df = pd.read_hdf(require_file(path))
    if "Correct" in df.columns:
        correct_col = "Correct"
    elif "passed" in df.columns:
        correct_col = "passed"
    else:
        raise ValueError(f"Expected a Correct or passed column in {path}")
    return df.groupby("task_id")[correct_col].mean()


def retok_only_correct_task_ids() -> list[str]:
    retok_correctness = correctness_by_task(RETOK_RESULTS)
    canon_correctness = correctness_by_task(CANON_RESULTS)
    common_task_ids = sorted(set(retok_correctness.index) & set(canon_correctness.index))
    return [
        task_id
        for task_id in common_task_ids
        if retok_correctness.loc[task_id] > 0 and canon_correctness.loc[task_id] == 0
    ]


df_all = load_pass_metric_comparison(RETOK_METRICS, CANON_METRICS)
df_correct = load_pass_metric_comparison(RETOK_CORRECT_METRICS, CANON_CORRECT_METRICS)
circled_task_ids = retok_only_correct_task_ids()

print(f"All-completion tasks: {len(df_all)}")
print(f"Correct-completion tasks: {len(df_correct)}")
print(f"Circled retok-only-correct tasks: {circled_task_ids}")

In [ ]:
def plot_pass_metric_panel(ax: plt.Axes, df: pd.DataFrame, title: str, circled_task_ids: list[str]) -> None:
    plot_df = df.dropna(
        subset=[
            "mean_syntactic_metric",
            "mean_syntactic_metric_passk",
            "std_error_retok",
            "std_error_passk",
        ]
    )

    above_diag = plot_df["mean_syntactic_metric_passk"] > plot_df["mean_syntactic_metric"]
    below_diag = plot_df["mean_syntactic_metric_passk"] < plot_df["mean_syntactic_metric"]
    on_diag = ~(above_diag | below_diag)

    for mask, color, label in [
        (above_diag, "royalblue", "Above diagonal"),
        (below_diag, "firebrick", "Below diagonal"),
        (on_diag, "black", "On diagonal"),
    ]:
        subset = plot_df.loc[mask]
        if subset.empty:
            continue
        ax.errorbar(
            x=subset["mean_syntactic_metric"],
            y=subset["mean_syntactic_metric_passk"],
            xerr=subset["std_error_retok"],
            yerr=subset["std_error_passk"],
            fmt="o",
            markersize=3,
            elinewidth=0.6,
            capsize=2,
            color=color,
            alpha=0.85,
            # label=label,
        )

    circled = plot_df.loc[plot_df.index.intersection(circled_task_ids)]
    if not circled.empty:
        ax.scatter(
            circled["mean_syntactic_metric"],
            circled["mean_syntactic_metric_passk"],
            marker="o",
            s=180,
            facecolors="none",
            edgecolors="firebrick",
            linewidths=1.8,
            label="pass@retok correct, pass@k not correct",
        )

    ax.plot([0, 1], [0, 1], "--", color="0.5", linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Mean syntactic metric (pass@retok)")
    ax.set_ylabel("Mean syntactic metric (pass@k)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4), constrained_layout=True)
plot_pass_metric_panel(axes[0], df_all, "All completions", circled_task_ids)
plot_pass_metric_panel(axes[1], df_correct, "Only correct completions", circled_task_ids)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=1, frameon=False, bbox_to_anchor=(0.25, 0.96))
fig.suptitle(MODEL_LABEL, y=1.03)

# save_figure(fig, "diversity_new")
plt.show()

In [ ]:
WITHIN_BETWEEN = new_metric_path(RESULT_DIR / "within_between.json")
WITHIN_BETWEEN_CORRECT = new_metric_path(RESULT_DIR / "within_between_correct.json")

In [ ]:
def load_within_between(path: Path) -> pd.DataFrame:
    required = [
        "mean_within_group_a_dissimilarity",
        "mean_between_dissimilarity",
        "mean_within_group_b_dissimilarity",
    ]
    df = read_jsonl(path)
    return df.dropna(subset=required).sort_values("mean_within_group_b_dissimilarity").reset_index(drop=True)


df_within_between = load_within_between(WITHIN_BETWEEN)
df_within_between_correct = load_within_between(WITHIN_BETWEEN_CORRECT)

print(f"All-completion tasks: {len(df_within_between)}")
print(f"Correct-completion tasks: {len(df_within_between_correct)}")

In [ ]:
def plot_d_canon_sorted_panel(ax: plt.Axes, df: pd.DataFrame, title: str) -> None:
    series = [
        ("mean_within_group_a_dissimilarity", "d(R)", "royalblue"),
        ("mean_between_dissimilarity", "d(R; C)", "firebrick"),
        ("mean_within_group_b_dissimilarity", "d(C)", "seagreen"),
    ]
    x = np.arange(len(df))
    for column, label, color in series:
        ax.plot(x, df[column], label=label, color=color, linewidth=2, alpha=0.9)

    ax.set_xlabel("d(C) sorted")
    ax.set_ylabel(r"Mean dissimilarity $\langle d\rangle$")
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True, sharey=True)
plot_d_canon_sorted_panel(axes[0], df_within_between, "All completions")
plot_d_canon_sorted_panel(axes[1], df_within_between_correct, "Only correct completions")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.08))

save_figure(fig, "olmo2_figure3_d_canon_sorted_new_metric")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.08, hspace=0.12, wspace=0.08)

plot_pass_metric_panel(axes[0, 0], df_all, "All completions", circled_task_ids)
plot_pass_metric_panel(axes[0, 1], df_correct, "Only correct completions", circled_task_ids)
plot_d_canon_sorted_panel(axes[1, 0], df_within_between, "All completions")
plot_d_canon_sorted_panel(axes[1, 1], df_within_between_correct, "Only correct completions")

for ax in axes.flat:
    ax.set_box_aspect(1)

top_handles, top_labels = axes[0, 0].get_legend_handles_labels()
if top_handles:
    fig.legend(top_handles, top_labels, loc="upper center", ncol=1, frameon=False, bbox_to_anchor=(0.23, 0.965))

bottom_handles, bottom_labels = axes[1, 0].get_legend_handles_labels()
if bottom_handles:
    fig.legend(bottom_handles, bottom_labels, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.23, 0.410))

fig.suptitle(MODEL_LABEL, y=0.995)

save_figure(fig, "olmo2_figure4_combined")
plt.show()


In [ ]:
def p_failure_by_task(path: Path) -> pd.Series:
    return 1.0 - correctness_by_task(path)


failure_all = pd.DataFrame({
    "p_failure_retok": p_failure_by_task(RETOK_RESULTS),
    "p_failure_passk": p_failure_by_task(CANON_RESULTS),
})

syntactic_failure_all = df_all.join(failure_all, how="inner").dropna(
    subset=[
        "mean_syntactic_metric",
        "mean_syntactic_metric_passk",
        "p_failure_retok",
        "p_failure_passk",
    ]
)

fig, ax = plt.subplots(figsize=(5.2, 4.6), constrained_layout=True)
ax.scatter(
    syntactic_failure_all["p_failure_passk"],
    syntactic_failure_all["mean_syntactic_metric_passk"],
    color="red",
    s=28,
    alpha=0.8,
    label="pass@k",
)
ax.scatter(
    syntactic_failure_all["p_failure_retok"],
    syntactic_failure_all["mean_syntactic_metric"],
    color="black",
    s=28,
    alpha=0.8,
    label="pass@retok",
)

passk_corr = syntactic_failure_all["p_failure_passk"].corr(
    syntactic_failure_all["mean_syntactic_metric_passk"]
)
retok_corr = syntactic_failure_all["p_failure_retok"].corr(
    syntactic_failure_all["mean_syntactic_metric"]
)
print(f"pass@k Pearson r: {passk_corr:.3f}")
print(f"pass@retok Pearson r: {retok_corr:.3f}")

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel(r"$P_{fail}$")
ax.set_ylabel("Mean syntactic metric")
ax.set_title("All completions: syntactic metric vs failure probability")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

save_figure(fig, "olmo2_all_completion_syntactic_metric_vs_pfailure")
plt.show()
